## **pydantic**

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# Step 1 — Define the shape of data you want
class MovieReview(BaseModel):
    title: str = Field(description="Name of the movie")
    rating: float = Field(description="Rating out of 10")
    pros: List[str] = Field(description="List of good things about the movie")
    cons: List[str] = Field(description="List of bad things about the movie")
    verdict: str = Field(description="One line final verdict")

# Step 2 — Create parser with your schema
parser = PydanticOutputParser(pydantic_object=MovieReview)

# Step 3 — Get format instructions from parser
# This tells the AI exactly what JSON structure to return
format_instructions = parser.get_format_instructions()
print(format_instructions)  # shows AI the exact schema to follow

## **Guiding in Prompts** without pydantic model

In [2]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate   # normal prompt
from langchain_core.prompts import ChatPromptTemplate # add tone also

load_dotenv()


model = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant",
    temperature=0.0
)

result = model.invoke("Tell me a joke. generate the output in key value pairs with folling keys  : setpup , punchline")
result.content

'Here\'s a joke for you:\n\n{\n  "setup": "A man walked into a library and asked the librarian, \'Do you have any books on Pavlov\'s dogs and Schrödinger\'s cat?\'",\n  "punchline": "The librarian replied, \'It rings a bell, but I\'m not sure if it\'s here or not.\'"\n}'

## **using pydantic Model**

In [6]:
from pydantic import BaseModel, Field

# create a schema
class llm_schema(BaseModel):
    key : str = Field(description="The key for the joke")
    value : str = Field(description="The value for the joke")

# obj = llm_schema(**{"setup" : "some setup", "punchline" : "some punchile"})
# obj.setup

llm_structure_Output = model.with_structured_output(llm_schema)
llm_structure_Output.invoke("Tell me a joke")

llm_schema(key='joke', value='Why don’t scientists trust atoms? Because they make up everything.')